In [ ]:
import aiohttp
import bs4
import orjson
import datetime
import dataclasses
import time
import os
import traceback
import asyncio
import typing
import aiofiles
import re

from aiohttp.client_exceptions import ConnectionTimeoutError

from credentials import RAPIDAPI_KEY, WEBHOOK_URL, PROXY

@dataclasses.dataclass
class Price24K:
    buy_bullion: int
    sell_bullion: int
    buy_ring: int
    sell_ring: int
    taken_on: datetime.datetime
    vendor: str = ""

In [2]:
async def yh_quote(session: aiohttp.ClientSession, symbol: str) -> tuple[float, float, float]:
    """:returns: price, change, change percent"""
    try:
        data = (await (
            await session.get(
                "https://yahoo-finance166.p.rapidapi.com/api/stock/get-price",
                headers={
                    "x-rapidapi-key": RAPIDAPI_KEY,
                    "x-rapidapi-host": "yahoo-finance166.p.rapidapi.com",
                    "Content-Type": "application/json",
                },
                params={"region": "US", "symbol": symbol},
            )
        ).json())["quoteSummary"]["result"][0]["price"]
        return data["regularMarketPrice"]["raw"], data["regularMarketChange"]["raw"], data["regularMarketChangePercent"]["raw"]
    except:
        return 0.0, 0.0, 0.0

async def binance_quote(session: aiohttp.ClientSession, symbol: str) -> tuple[float, float, float]:
    req = await session.get(f"https://api.binance.us/api/v3/ticker/24hr?symbol={symbol}")
    try:
        data = await req.json()
        return float(data["lastPrice"]), float(data["priceChange"]), float(data["priceChangePercent"])
    except:
        print(f"[!] fetching binance symbol {symbol} failed: {req.status} {req.method} {req.url}")
        print(await req.text())



In [3]:
async def fetch_vcb_vnd(session):
    vcb_api_output = await (
        await session.get(
            "https://portal.vietcombank.com.vn/Usercontrols/TVPortal.TyGia/pXML.aspx"
        )
    ).text()
    matches = VCB_VND_MATCH_PATTERN.finditer(
        vcb_api_output
    )

    for match in matches:
        for group in match.groups():
            return float(group.replace(",", ""))
    else:
        print(vcb_api_output)
        raise ConnectionError("Cannot fetch VCB VND price.")

In [4]:
async def fetch_coinbase(session: aiohttp.ClientSession, instrument: str) -> float:
    req = await session.get(f"https://api.international.coinbase.com/api/v1/instruments/{instrument}")
    try:
        data = await req.json()
        return float(data["quote"]["mark_price"])
    except:
        print(f"[!] fetching coinbase instrument {instrument} failed: {req.status} {req.method} {req.url}")
        print(await req.text())

In [5]:
ONE_MILLION = 1_000_000

# Converts 100 troy ounces (COMEX contract size) to 1 lượng
CONVERSION_FACTOR = 1 / 0.0311034768 * 3 / 80 * 1 / ONE_MILLION

VCB_VND_MATCH_PATTERN = re.compile(
    r'CurrencyCode="USD"[\s\S]*?Sell="([^"]+)"', flags=re.MULTILINE
)

async def tail(filename, n=1, chunk_size=2048) -> list[str]:
    async with aiofiles.open(filename, "rb") as f:
        await f.seek(0, os.SEEK_END)
        pointer_pos = await f.tell()
        buffer = b""

        while pointer_pos > 0 and buffer.count(b"\n") <= n:
            pointer_pos = max(0, pointer_pos - chunk_size)
            await f.seek(pointer_pos)

            bytes_to_read = (
                (pointer_pos + chunk_size) if pointer_pos == 0 else chunk_size
            )
            chunk = await f.read(bytes_to_read)

            buffer = chunk + buffer

        lines = buffer.split(b"\n")

        if lines and lines[-1] == b"":
            lines.pop()

        return [line.decode("utf-8") for line in lines[-n:]]


async def compare_price(
    current: list[Price24K],
) -> dict[str, tuple[float, float] | None]:
    t = (await tail(os.path.join("data", "master.jsonl"), n=5))[0].strip()
    data = [Price24K(**i) for i in orjson.loads(t)["prices"]]
    output = {}
    for new_price in current:
        for old_price in data:
            if old_price.vendor == new_price.vendor:
                output[new_price.vendor] = (
                    new_price.sell_bullion - old_price.sell_bullion,
                    (new_price.sell_bullion - old_price.sell_bullion)
                    / old_price.sell_bullion,
                )
    return output

class Announce:
    @staticmethod
    async def price(
        session: aiohttp.ClientSession, data: list[Price24K]
    ) -> aiohttp.ClientResponse:
        comex_price = await fetch_coinbase(session, "GOLD-PERP")
        paxg_price = await binance_quote(session, "PAXGUSDT")
        comparison = await compare_price(data)
        vcb_vnd = await fetch_vcb_vnd(session)
        print("[*] VCB fx rate: ", vcb_vnd)
        print("[*] COMEX", comex_price)
        print("[*] PAXG", paxg_price[0])
        return await session.post(
            url=WEBHOOK_URL,
            params={"with_components": "true"},
            json={
                "flags": 32768,
                "components": [
                    {
                        "type": 17,
                        "accent_color": None,
                        "spoiler": False,
                        "components": [
                            {
                                "type": 10,
                                "content": "## ShitCapital XAUVND Tracker Report",
                            },
                            {
                                "type": 10,
                                "content": f"Báo cáo lúc <t:{int(time.time())}:F>",
                            },
                            {
                                "type": 10,
                                "content": f"Giá theo triệu đồng trên một lượng. Tỉ giá USD Vietcombank.",
                            },
                            {"type": 14, "divider": True, "spacing": 1},
                            {
                                "type": 10,
                                "content": f"* **COMEX**:  `{comex_price} USD`, `{(comex_price * vcb_vnd * CONVERSION_FACTOR):.2f}M VND`",
                            },
                            {
                                "type": 10,
                                "content": f"* **PAXG**:  `{paxg_price[0]} USDC` {"▲" if paxg_price[1] >= 0 else "▼"}{paxg_price[1]:.2f} ({paxg_price[2]:.2f}%), `{(paxg_price[0] * vcb_vnd * CONVERSION_FACTOR):.2f}M VND`",
                            },
                            *[
                                {
                                    "type": 10,
                                    "content": (
                                        f"* **{d.vendor}**:  bid `{d.buy_bullion/ONE_MILLION}`, ask `{d.sell_bullion/ONE_MILLION}` {"▲" if comparison[d.vendor][0] >= 0 else "▼"}{comparison[d.vendor][0]} ({comparison[d.vendor][0]:.2f}%), spread `{(d.sell_bullion - d.buy_bullion)/ONE_MILLION}`"  # type: ignore
                                        if comparison[d.vendor] is not None
                                        else f"* **{d.vendor}**:  bid `{d.buy_bullion/ONE_MILLION}`, ask `{d.sell_bullion/ONE_MILLION}`, spread `{(d.sell_bullion - d.buy_bullion)/ONE_MILLION}`"
                                    ),
                                }
                                for d in data
                            ],
                            {"type": 14, "divider": True, "spacing": 1},
                            {
                                "type": 10,
                                "content": "-# <:checkmark:1263823949079384095> Được kiểm chứng bởi Mr. Forest. Xem [Quyết định 67-2026/NQ-TW](https://example.com).",
                            },
                            {
                                "type": 10,
                                "content": "-# <:checkmark:1263823949079384095> Đã kiểm chứng bởi Warren Buffet. [Tìm hiểu thêm](https://example.com).",
                            },
                        ],
                    }
                ],
            },
        )

In [6]:
async def sjc_today(session: aiohttp.ClientSession) -> Price24K:
    buy_bullion: int = 0
    sell_bullion: int = 0
    buy_ring: int = 0
    sell_ring: int = 0
    # TODO - temporary countermeasure as SJC uses Cloudflare
    # with requests.Session() as s:
    #     data = s.get("https://sjc.com.vn/GoldPrice/Services/PriceService.ashx")
    #     for i in data["data"]:
    #         if i["TypeName"] == "Vàng SJC 1L, 10L, 1KG" and i["BranchName"] == "Hồ Chí Minh":
    #             buy_bullion = int(i["BuyValue"])
    #             sell_bullion = int(i["SellValue"])
    #         if i["TypeName"] == "Vàng nhẫn SJC 99,99% 1 chỉ, 2 chỉ, 5 chỉ" and i["BranchName"] == "Hồ Chí Minh":
    #             buy_ring = int(i["BuyValue"])
    #             sell_ring = int(i["SellValue"])
    #     return Price24K(
    #         buy_bullion, sell_bullion, buy_ring, sell_ring, taken_on=datetime.datetime.now(), vendor="SJC"
    #     )
    data = await (await session.get("https://edge-api.pnj.io/ecom-frontend/v1/get-gold-price?zone=00")).json()
    for i in data["data"]:
        if i["masp"] == "SJC":
            sell_ring = sell_bullion = i["giaban"]*10000
            buy_ring = buy_bullion = i["giamua"]*10000
    return Price24K(
        buy_bullion, sell_bullion, buy_ring, sell_ring, taken_on=datetime.datetime.now(), vendor="SJC"
    )

In [7]:
async def doji_today(session: aiohttp.ClientSession) -> Price24K:
    req = await session.get("https://giavang.doji.vn/")
    soup = bs4.BeautifulSoup(await req.text())
    buy_bullion, sell_bullion, buy_ring, sell_ring = [int(list(i.children)[0].text.replace(",", "")) * 10_000 for i in soup.select(".goldprice-view .goldprice-td")][2:6]

    return Price24K(
        buy_bullion, sell_bullion, buy_ring, sell_ring, taken_on=datetime.datetime.now(), vendor="DOJI"
    )

In [8]:
async def pnj_today(session: aiohttp.ClientSession) -> Price24K:
    buy_bullion: int = 0
    sell_bullion: int = 0
    buy_ring: int = 0
    sell_ring: int = 0
    data = await (await session.get("https://edge-api.pnj.io/ecom-frontend/v1/get-gold-price?zone=00")).json()
    for i in data["data"]:
        if i["masp"] == "N24K":
            sell_ring = i["giaban"]*10000
            buy_ring = i["giamua"]*10000
        if i["masp"] == "KB":
            sell_bullion = i["giaban"]*10000
            buy_bullion = i["giamua"]*10000
    return Price24K(
        buy_bullion, sell_bullion, buy_ring, sell_ring, taken_on=datetime.datetime.now(), vendor="PNJ"
    )

In [ ]:
async def btmc_today(session: aiohttp.ClientSession) -> Price24K:
    soup = bs4.BeautifulSoup(await (await session.get("https://btmc.vn/gia-vang-theo-ngay.html", proxy=PROXY)).text(), "html.parser")
    bullion = list(list(soup.find_all(class_="bd_price_home")[0].children)[1].children)[3]  # type: ignore
    ring = list(list(soup.find_all(class_="bd_price_home")[0].children)[1].children)[4]  # type: ignore
    return Price24K (
        taken_on=datetime.datetime.now(),
        buy_bullion = int(list(bullion.children)[7].getText().strip()) * 10000,
        sell_bullion = int(list(bullion.children)[9].getText().strip()) * 10000,
        buy_ring = int(list(ring.children)[5].getText().strip()) * 10000,
        sell_ring = int(list(ring.children)[7].getText().strip()) * 10000,
        vendor="BTMC"
    )

In [ ]:
async def btmh_today(session: aiohttp.ClientSession) -> Price24K:
    soup = bs4.BeautifulSoup(await (await session.get("https://baotinmanhhai.vn/gia-vang-hom-nay", ssl=False, proxy=PROXY)).text(), "html.parser")
    buy_bullion: int = 0
    sell_bullion: int = 0
    buy_ring: int = 0
    sell_ring: int = 0

    for i in list(filter(lambda x: True if x.select(".items-center") else False,soup.select("div.flex div.grid")))[:-3]:
        if i.contents[0].contents[1].text == "Nhẫn Tròn ép vỉ (Kim Gia Bảo ) 24K (999.9)":
            sell_ring = int(i.contents[1].contents[0].contents[0].text.replace(".", "")) * 10
            buy_ring = int(i.contents[2].contents[0].contents[0].text.replace(".", "")) * 10

        if i.contents[0].contents[1].text == "Đồng vàng Kim Gia Bảo hoa sen":
            sell_bullion = int(i.contents[1].contents[0].contents[0].text.replace(".", "")) * 10
            buy_bullion = int(i.contents[2].contents[0].contents[0].text.replace(".", "")) * 10

    return Price24K(    
        taken_on=datetime.datetime.now(),
        buy_bullion=buy_bullion,
        sell_bullion=sell_bullion,
        buy_ring=buy_ring,
        sell_ring=sell_ring,
        vendor="BTMH"
    )

In [11]:
async def pqg_today(session: aiohttp.ClientSession) -> Price24K:
    soup = bs4.BeautifulSoup(await (await session.get("https://phuquygroup.vn/")).text(), "html.parser")
    price_table = list(soup.select_one("#priceList table").select("tr"))  # type: ignore
    return Price24K (
        taken_on=datetime.datetime.now(),
        buy_ring = int(price_table[2].select_one("td.buy-price").text.strip().replace(",", "")) * 10,  # type: ignore
        sell_ring = int(price_table[2].select_one("td.sell-price").text.strip().replace(",", "")) * 10,  # type: ignore
        buy_bullion = int(price_table[3].select_one("td.buy-price").text.strip().replace(",", "")) * 10,  # type: ignore
        sell_bullion = int(price_table[3].select_one("td.sell-price").text.strip().replace(",", "")) * 10,  # type: ignore
        vendor="PQG",
    )

In [12]:
async def mihong_today(session: aiohttp.ClientSession) -> Price24K:
    buy_bullion = 0
    sell_bullion = 0
    buy_ring = 0
    sell_ring = 0
    data = await (await session.get("https://api.mihong.vn/v1/gold-prices?market=domestic")).json()
    for i in data:
        if i["code"] == "999":
            buy_bullion = buy_ring = i["buyingPrice"] * 10
            sell_ring = sell_bullion = i["sellingPrice"] * 10


    return Price24K(
        taken_on=datetime.datetime.now(),
        buy_bullion=buy_bullion,
        sell_bullion=sell_bullion,
        buy_ring=buy_ring,
        sell_ring=sell_ring,
        vendor="MHG"
    )

In [13]:
async def ngoctham_today(session: aiohttp.ClientSession) -> Price24K:
    s = bs4.BeautifulSoup(await (await session.get("https://ngoctham.com/bang-gia-vang/")).text())
    buy_bullion = 0
    sell_bullion = 0
    buy_ring = 0
    sell_ring = 0

    for i in list(s.select("table.table tr"))[3:]:
        if list(i.find_all("td", attrs={"class": "name-gold"}))[0].text == "Vàng Ta 999.9":
            buy_bullion = int(list(i.find_all("td", attrs={"class": "price-buy"}))[0].text.replace(".", "")) * 10
            sell_bullion = int(list(i.find_all("td", attrs={"class": "price-sell"}))[0].text.replace(".", "")) * 10
        if list(i.find_all("td", attrs={"class": "name-gold"}))[0].text == "Nhẫn 999.9":
            buy_ring = int(list(i.find_all("td", attrs={"class": "price-buy"}))[0].text.replace(".", "")) * 10
            sell_ring = int(list(i.find_all("td", attrs={"class": "price-sell"}))[0].text.replace(".", "")) * 10

    return Price24K(
        taken_on=datetime.datetime.now(),
        buy_bullion=buy_bullion,
        sell_bullion=sell_bullion,
        buy_ring=buy_ring,
        sell_ring=sell_ring,
        vendor="NTH"
    )

In [14]:
async def do_fetch(session: aiohttp.ClientSession, fetch_method: typing.Callable[[aiohttp.ClientSession], typing.Awaitable[Price24K]]) -> tuple[Price24K, str] | None:
    async def fetch(tries = 10) -> Price24K | None:
        try:
            return await fetch_method(session)
        except ConnectionTimeoutError:
            return
        except:
            if tries > 0:
                print(f"[!] fetching failed for {fetch_method.__name__}. Retrying after 5s (attempt {10 - tries + 1}/10).")
                traceback.print_exc()
                time.sleep(5)
                return await fetch(tries-1)
            raise ConnectionError(f"Cannot fetch {fetch_method.__name__}")
            
    d = await fetch()
    if d:
        print(f"[*] fetched    {d.vendor:>4}  {d.buy_bullion}      {d.sell_bullion}       {d.buy_ring}     {d.sell_ring}")
        return d, f"{d.taken_on.isoformat()},{d.vendor},{d.buy_bullion},{d.sell_bullion},{d.buy_ring},{d.sell_ring}\n"
    return None

async def fetch_all(session: aiohttp.ClientSession) -> list[Price24K]:
    results = []
    print(f"            vendor  buy bullion    sell bullion    buy ring      sell ring")
    #       [*] fetched    SJC  173000000      173000000       173000000     173000000
    r = await asyncio.gather(*[
        do_fetch(session, i) for i in (sjc_today, doji_today, pnj_today, btmc_today, btmh_today, pqg_today, mihong_today, ngoctham_today)
    ], return_exceptions=True)
    r = [i for i in r if isinstance(i, tuple) and len(i) == 2]
    if len(r) != 0:
        results, csvdata = zip(*r)
        results, csvdata = list(results), "".join(csvdata)
        async with aiofiles.open(os.path.join("data", "snapshotdb.csv"), "a", encoding="utf8") as f:
            await f.write(csvdata)
    else:
        print("[*] No results found. All methods may have failed.")
    return results

async def run_cron():
    async with aiohttp.ClientSession() as session:
        prices = await fetch_all(session)
        async with aiofiles.open(os.path.join("data", "master.jsonl"), "a", encoding="utf8") as f:
            f.buffer.write(orjson.dumps({
                "takenOn": datetime.datetime.now().isoformat(),
                "prices": prices
            }))
            f.buffer.write(b"\n")
            await f.flush()
        wh = await Announce.price(session, prices)
        if wh.status == 204:
            print("[*] Webhook sent successfully!")
        else:
            print(f"[*] Failed to send webhook: {wh.status}, {await wh.text()}")

In [15]:
await run_cron()

            vendor  buy bullion    sell bullion    buy ring      sell ring
[*] fetched     MHG  146300000      148300000       146300000     148300000
[*] fetched     PQG  145000000      148000000       145000000     148000000
[*] fetched     SJC  145500000      148500000       145500000     148500000
[*] fetched     PNJ  145500000      148500000       145500000     148500000
[*] fetched    BTMC  144300000      148000000       144300000     148000000
[*] fetched     NTH  133500000      137000000       133500000     137000000
[*] fetched    BTMH  144300000      148300000       144300000     148300000
[*] fetched    DOJI  145500000      148500000       145500000     148500000
[*] VCB fx rate:  26454.0
[*] COMEX 4076.9
[*] PAXG 4079.94
[*] Webhook sent successfully!
